# LSTM & Trainer Class

In [9]:
import torch
from torch import nn
import tqdm.auto

c:\Users\Ruben\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class LSTM(nn.Module):
    def __init__(self, k: int,
                 hidden_size: int = 64,
                 num_layers: int = 2,
                 dropout: float = 0.2,
                 batch_first: bool = True):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size  = 5 + 2 * k,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout,
            batch_first = batch_first
        )

        self.mu       = nn.Linear(hidden_size, 1)
        self.sigma    = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Softplus()
        )
        self.momentum = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]

        mu    = self.mu(last)
        sigma = self.sigma(last)
        m_hat = self.momentum(last)

        return mu, sigma, m_hat

In [ ]:
def nll_loss(mu, sigma, target):
    return torch.log(sigma) + (target - mu) ** 2 / (2 * sigma ** 2)

def sign_bce_loss(mu, sigma, target):
    p_hat = torch.sigmoid(mu / sigma)
    y_sign = (target > 0).float
    return nn.BCELoss(reduction="none")(p_hat, y_sign)

def crps_loss(mu, sigma, target):
    dist  = torch.distributions.Normal(mu, sigma)
    u     = (target - mu) / sigma
    phi   = torch.exp(dist.log_prob(target)) 
    Phi   = dist.cdf(target) 
    return sigma * (2 * phi + u * (2 * Phi - 1) - 1 / torch.tensor(torch.pi).sqrt())

def momentum_loss(m_hat, m_target):
    return (m_target - m_hat) ** 2

def total_loss(mu, sigma, m_hat, target_r, target_m, weights_r, weights_m):
    return_loss   = nll_loss(mu, sigma, target_r) + 0.3 * sign_bce_loss(mu, sigma, target_r) + 0.2 * crps_loss(mu, sigma, target_r)
    momentum      = momentum_loss(m_hat, target_m)

    return (return_loss * weights_r).mean() + 0.3 * (momentum * weights_m).mean()

class LSTM_Trainer():
    def __init__(self, k: int,
                 hidden_size: int=64,
                 num_layers: int=2,
                 dropout: int=0.2,
                 batch_first: bool=True,
                 epochs: int=200):
        model = LSTM(k=k,
                     hidden_size=hidden_size,
                     num_layers=num_layers,
                     dropout=dropout,
                     batch_first=batch_first)
        self.optimizer = torch.optim.Adam(params=model.parameters(), 
                                          lr=1e-3)

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode     = 'min',
            factor   = 0.5,   
            patience = 10, 
        )

        self.epochs = epochs

    def train(self, dataloader):
        self.model.train()
        total_loss = 0
        for X, r, m, w_r, w_m in dataloader:
            self.optimizer.zero_grad()

            mu, sigma, m_hat = self.model(X)

            loss = total_loss(mu, sigma, m_hat, r, m, w_r, w_m)
            total_loss += loss.item()

            self.optimizer.step()
            loss.backward()

        total_loss /= len(dataloader)

        return total_loss

    def evaluate(self, dataloader):
        self.model.eval()

        total_loss = 0
        with torch.inference_mode():
            for X, r, m, w_r, w_m in dataloader:
                mu, sigma, m_hat = self.model(X)

                loss = total_loss(mu, sigma, m_hat, r, m, w_r, w_m)
                total_loss += loss.item()

        total_loss /= len(dataloader)

        return total_loss

                
    